# CUDA 설치 확인

In [1]:
import torch
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")
print(f"현재 GPU 이름: {torch.cuda.get_device_name(0)}")

PyTorch 버전: 2.5.1+cu121
CUDA 사용 가능 여부: True
현재 GPU 이름: NVIDIA GeForce GTX 1070


# 라이브러리 임포트

In [23]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 29.3 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 28.7 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [24]:
import os
import json
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence
from tqdm.notebook import tqdm
from sklearn.model_selection import KFold

# GTX 1070 사용을 위한 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 장치: {device}")

현재 사용 중인 장치: cuda


# 정답 데이터 로드

In [12]:
# 상대경로 설정
label_dir = "./Data/01/morpheme/"
label_paths = glob.glob(os.path.join(label_dir, "*_morpheme.json"))

def load_data(label_paths):
    labels = []
    for path in label_paths:
        with open(path, 'r', encoding='utf-8') as f:
            content = json.load(f)
            
            # 1. 원본 영상 파일명 (morpheme.json에서 추출)
            full_name = content['metaData']['name'] # 예: NIA_SL_WORD0001_REAL01_D.mp4
            base_name = full_name.replace('.mp4', '') # 확장자 제거
            
            # 2. 정답 단어 및 시간 정보[cite: 1]
            word = content['data'][0]['attributes'][0]['name']
            start_t = content['data'][0]['start']
            end_t = content['data'][0]['end']
            
            labels.append({
                'folder_name': base_name, # 키포인트 폴더 이름과 일치
                'word': word,
                'start': start_t,
                'end': end_t
            })
    return labels

label_list = load_data(label_paths)

# 리스트에서 'word'만 추출하여 집합으로 변환
words = set([item['word'] for item in label_list])


df = pd.DataFrame(label_list)
print(f"총 {len(df)}개의 정답지를 불러왔습니다.")
display(df.head(10)) # 데이터가 잘 들어왔는지 표로 확인

총 1000개의 정답지를 불러왔습니다.


,folder_name,word,start,end
0,NIA_SL_WORD0006_REAL01_D,독신,1.248,2.778
1,NIA_SL_WORD0006_REAL01_F,독신,1.248,2.778
2,NIA_SL_WORD0006_REAL01_L,독신,1.248,2.778
3,NIA_SL_WORD0006_REAL01_R,독신,1.248,2.778
4,NIA_SL_WORD0006_REAL01_U,독신,1.248,2.778
5,NIA_SL_WORD0006_SYN01_D,독신,0.377,1.547
6,NIA_SL_WORD0006_SYN01_F,독신,0.377,1.547
7,NIA_SL_WORD0006_SYN01_L,독신,0.377,1.547
8,NIA_SL_WORD0006_SYN01_R,독신,0.377,1.547
9,NIA_SL_WORD0006_SYN01_U,독신,0.377,1.547


# 단어 100개 추출 및 저장
실행 X

In [9]:
import os
import shutil
import json
import random
from tqdm import tqdm
from collections import defaultdict

# 1. 원본 디렉토리 경로 설정
path_real_coord = "./Train/01/"                # REAL 좌표 폴더
path_real_morph = "./Train/morpheme/01/"       # REAL 정답 JSON
path_syn_coord = "./Train/WORD/keypoint/01/"    # SYN 좌표 폴더
path_syn_morph = "./Train/WORD/morpheme/01/"    # SYN 정답 JSON

# 2. 내보낼 경로 설정
export_base = "./Export_Team_Data/01/"
dirs = ["REAL", "SYN", "morpheme"]
for d in dirs:
    os.makedirs(os.path.join(export_base, d), exist_ok=True)

# 3. 정답 JSON을 열어 단어별로 파일명/폴더명을 수집하는 함수
def scan_morpheme(morph_path):
    word_map = defaultdict(list)
    # 해당 경로의 모든 _morpheme.json 파일 탐색
    json_files = glob.glob(os.path.join(morph_path, "*_morpheme.json"))
    
    for j_path in json_files:
        try:
            with open(j_path, 'r', encoding='utf-8') as f:
                content = json.load(f)
                # JSON 내부의 실제 단어 이름 확인
                word_name = content['data'][0]['attributes'][0]['name']
                # 파일명에서 폴더명 추출 (확장자 제외)
                base_name = content['metaData']['name'].replace('.mp4', '')
                word_map[word_name].append({
                    'folder': base_name,
                    'json_path': j_path
                })
        except:
            continue
    return word_map

print("🔍 REAL/SYN 정답지 분석 중...")
real_data_map = scan_morpheme(path_real_morph)
syn_data_map = scan_morpheme(path_syn_morph)

# 4. 공통 단어 중 5개 각도가 모두 있는 단어 필터링
common_words = [
    w for w in real_data_map 
    if w in syn_data_map and len(real_data_map[w]) == 5 and len(syn_data_map[w]) == 5
]

print(f"✅ 교집합 단어 중 완벽한 세트(각도 5개씩)를 가진 단어: {len(common_words)}개")

# 5. 100개 랜덤 선정
random.seed(42)
selected_100 = random.sample(common_words, 100)

# 6. 데이터 복사 (4개 경로에서 데이터 수집)
for word in tqdm(selected_100, desc="데이터 수집 및 분류 중"):
    # REAL 데이터 복사
    for item in real_data_map[word]:
        # 좌표 폴더 복사 (Train/01/ -> Export/REAL/)
        src_folder = os.path.join(path_real_coord, item['folder'])
        if os.path.exists(src_folder):
            shutil.copytree(src_folder, os.path.join(export_base, "REAL", item['folder']), dirs_exist_ok=True)
        # 정답 파일 복사 (Train/morpheme/01/ -> Export/morpheme/)[cite: 3]
        shutil.copy2(item['json_path'], os.path.join(export_base, "morpheme"))

    # SYN 데이터 복사
    for item in syn_data_map[word]:
        # 좌표 폴더 복사 (Train/WORD/keypoint/01/ -> Export/SYN/)
        src_folder = os.path.join(path_syn_coord, item['folder'])
        if os.path.exists(src_folder):
            shutil.copytree(src_folder, os.path.join(export_base, "SYN", item['folder']), dirs_exist_ok=True)
        # 정답 파일 복사 (Train/WORD/morpheme/01/ -> Export/morpheme/)[cite: 3]
        shutil.copy2(item['json_path'], os.path.join(export_base, "morpheme"))

print(f"🚀 완료! '{export_base}' 폴더에 100단어의 모든 데이터가 정리되었습니다.")

🔍 REAL/SYN 정답지 분석 중...
✅ 교집합 단어 중 완벽한 세트(각도 5개씩)를 가진 단어: 453개


데이터 수집 및 분류 중: 100%|█████████████████████████████████████████████████████████| 100/100 [54:33<00:00, 32.73s/it]

🚀 완료! './Export_Team_Data/01/' 폴더에 100단어의 모든 데이터가 정리되었습니다.


# 데이터 가공 (키포인트 추출)

In [18]:
def extract_keypoints(frame_path):
    with open(frame_path, 'r') as f:
        data = json.load(f)
    
    # 1. 사람이 감지되지 않았을 경우를 대비해 0으로 채워진 배열 생성
    if not data['people']:
        return np.zeros(132)
    
    people_data = data['people']
    
    if isinstance(people_data, list):
        person = people_data[0]
    else:
        # people 자체가 딕셔너리인 경우 (일부 버전의 특징)
        person = people_data
    
    # 2. 각 부위별 키포인트 추출 (x, y, confidence 순서)
    try:
        # 왼손: 21개 점 * 3 = 63개 수치
        left_hand = np.array(person['hand_left_keypoints_2d']) 
        # 오른손: 21개 점 * 3 = 63개 수치
        right_hand = np.array(person['hand_right_keypoints_2d'])
        # 포즈: 25개 점 * 3 = 75개 수치 (여기서 필요한 부분만 추출)
        pose = np.array(person['pose_keypoints_2d'])
        
        # 3. 132개 차원 맞추기 전략
        # 방법 A: 양손 전체(63+63=126) + 포즈 중 눈/귀 등을 제외한 어깨 등 주요 2개 점(6개 수치)
        # 수어에서 중요한 것은 손의 위치와 모양이므로 손 데이터에 집중합니다.
        
        # confidence(신뢰도)를 제외하고 (x, y)만 쓴다면 점의 개수를 더 늘릴 수 있습니다.
        # 여기서는 구조 유지를 위해 x, y, c를 모두 포함하여 132개를 맞춥니다.
        combined = np.concatenate([
            left_hand,           # 63개
            right_hand,          # 63개
            pose[:6]             # 포즈 데이터 중 앞부분 2개 점 (코, 목 등) 6개
        ]) # 총 132개
        
        return combined
        
    except Exception as e:
        # 도저히 안 읽히는 파일은 0으로 채운 값을 반환해서 학습 흐름 유지
        return np.zeros(132)

In [19]:
# 1. 정리된 데이터 경로 설정
export_root = "./Data/01/"
path_real = os.path.join(export_root, "REAL")
path_syn = os.path.join(export_root, "SYN")
path_morph = os.path.join(export_root, "morpheme")

def load_sign_dataset():
    X_list = []
    y_list = []
    word_to_idx = {}
    idx_counter = 0

    # 모든 정답 JSON 파일 리스트 확보
    morph_files = glob.glob(os.path.join(path_morph, "*.json"))
    
    for j_path in tqdm(morph_files, desc="데이터셋 로드 중"):
        with open(j_path, 'r', encoding='utf-8') as f:
            meta = json.load(f)
            
            # 단어 및 시간 정보 추출
            word = meta['data'][0]['attributes'][0]['name']
            start_t = meta['data'][0]['start']
            end_t = meta['data'][0]['end']
            
            # 폴더명 (파일명에서 .mp4 제거)
            folder_name = meta['metaData']['name'].replace('.mp4', '')
            
            # REAL 또는 SYN 폴더 중 어디에 있는지 확인
            if "REAL" in folder_name:
                coord_dir = os.path.join(path_real, folder_name)
            else:
                coord_dir = os.path.join(path_syn, folder_name)

            if not os.path.exists(coord_dir):
                continue

            # 단어 라벨링 (텍스트 -> 숫자)
            if word not in word_to_idx:
                word_to_idx[word] = idx_counter
                idx_counter += 1
            
            # 실제 좌표 데이터 로드 (프레임별 JSON 읽기)
            # 30fps 가정 (현장에 맞춰 수정 가능)
            fps = 30
            start_idx = int(start_t * fps)
            end_idx = int(end_t * fps)
            
            json_frames = sorted(glob.glob(os.path.join(coord_dir, "*.json")))
            selected_frames = json_frames[start_idx : end_idx]
            
            sequence = []
            for frame_path in selected_frames:
                with open(frame_path, 'r') as ff:
                    frame_data = json.load(ff)
                    points = extract_keypoints(frame_path)
                    sequence.append(points)
            
            if len(sequence) > 0:
                X_list.append(torch.FloatTensor(np.array(sequence)))
                y_list.append(word_to_idx[word])

    # 2. 시퀀스 패딩 (모든 영상의 길이를 제일 긴 것에 맞춤)
    X_padded = pad_sequence(X_list, batch_first=True)
    y_tensor = torch.LongTensor(y_list)
    
    return X_padded, y_tensor, word_to_idx

# 데이터 실행
X, y, word_dic = load_sign_dataset()
print(f"✅ 최종 학습 데이터 모양: {X.shape}") # (총데이터수, 최대프레임, 132)
print(f"✅ 총 클래스(단어) 수: {len(word_dic)}")

# 3. 데이터셋 생성
dataset = TensorDataset(X, y)

데이터셋 로드 중:   0%|          | 0/1000 [00:00<?, ?it/s]

✅ 최종 학습 데이터 모양: torch.Size([1000, 86, 132])
✅ 총 클래스(단어) 수: 100


# 모델 학습, 평가

In [20]:
# 모델 구조
class SignLanguageClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super(SignLanguageClassifier, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # x: (Batch, Seq_len, Input_dim)
        out, _ = self.gru(x)
        # 마지막 프레임의 결과만 사용해서 분류
        out = self.fc(out[:, -1, :])
        return out

In [28]:
# 1. 전체 데이터 준비 (이전 단계에서 로드한 X, y 텐서)
# X: (1000, Max_Seq, 132), y: (1000,)
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

input_size = 132
hidden_size = 64
output_dim = len(word_dic)
num_layers = 2

# 결과 저장용 리스트
fold_results = []

# 2. K-Fold 루프 시작
for fold, (train_ids, val_ids) in enumerate(kf.split(X)):
    print(f"--- Fold {fold + 1} / {k_folds} ---")
    
    # 데이터셋 분리
    train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
    val_subsampler = torch.utils.data.SubsetRandomSampler(val_ids)
    
    train_loader = DataLoader(dataset, batch_size=16, sampler=train_subsampler)
    val_loader = DataLoader(dataset, batch_size=16, sampler=val_subsampler)
    
    # 모델 초기화 (매 폴드마다 새로 시작해야 함)
    model = SignLanguageClassifier(input_size, hidden_size, 
                                   output_dim, num_layers).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    # 실제 학습 루프
    num_epochs = 100
    pbar = tqdm(range(num_epochs), desc="전체 학습 진행도")
    for epoch in pbar:
        model.train()
        train_loss = 0
        for batch_X, batch_y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            output = model(batch_X)
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
    
        print(f"Epoch {epoch+1} Loss: {train_loss/len(train_loader):.4f}")
        
    # 각 폴드에서 가장 성능이 좋았던 가중치를 저장하거나 성능 기록
    
    # 검증 정확도 측정
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
    
    acc = 100 * correct / total
    print(f"Fold {fold + 1} Accuracy: {acc:.2f}%")
    fold_results.append(acc)

# 3. 최종 평균 성능 확인
print(f"\n✅ 평균 검증 정확도: {np.mean(fold_results):.2f}% (+/- {np.std(fold_results):.2f})")

--- Fold 1 / 5 ---


전체 학습 진행도:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1/100:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1 Loss: 4.6172


Epoch 2/100:   0%|          | 0/50 [00:00<?, ?it/s]

...

Epoch 98 Loss: 2.5679


Epoch 99/100:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 99 Loss: 2.5213


Epoch 100/100:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 100 Loss: 2.4824
Fold 5 Accuracy: 8.00%

✅ 평균 검증 정확도: 7.90% (+/- 1.28)


# 모델 저장

In [9]:
save_path = "./sign_model_v1.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'word_to_idx': word_to_idx
}, save_path)
print(f"모델이 {save_path}에 저장되었습니다.")

모델이 ./sign_model_v1.pth에 저장되었습니다.


# 모델 불러오기

In [10]:
# 1. 모델 구조 다시 정의 (저장할 때와 동일해야 함)
# 위에서 이미 정의했다면 생략 가능하지만, 새 파일에서 불러올 땐 필요합니다.
model_load = SignLanguageClassifier(input_size, hidden_size, num_classes, num_layers).to(device)

# 2. 저장된 파일 불러오기
checkpoint = torch.load("./sign_model_v1.pth", map_location=device)

# 3. 모델 가중치 복구
model_load.load_state_dict(checkpoint['model_state_dict'])

# 4. 단어 사전 복구
loaded_word_to_idx = checkpoint['word_to_idx']
# 반대로 숫자에서 단어를 찾는 사전도 만들어둡니다.
idx_to_word = {i: word for word, i in loaded_word_to_idx.items()}

model_load.eval() # 추론 모드로 전환 (Dropout 등이 비활성화됨)
print("모델 및 단어 사전 불러오기 완료!")

모델 및 단어 사전 불러오기 완료!


/tmp/ipykernel_942/1973582600.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("./sign_model_v1.pth", map_location=device)


# 불러온 모델 테스트

In [11]:
# 검증 데이터셋에서 샘플 하나 꺼내오기
test_input, test_label = val_dataset[0] 
test_input = test_input.unsqueeze(0).to(device) # 배치를 위한 차원 추가 (1, Seq, 132)

# 모델 예측
with torch.no_grad():
    output = model_load(test_input)
    _, predicted_idx = torch.max(output, 1)

# 결과 출력
pred_word = idx_to_word[predicted_idx.item()]
real_word = idx_to_word[test_label.item()]

print(f"모델의 예측: {pred_word}")
print(f"실제 정답: {real_word}")

모델의 예측: 발가락
실제 정답: 발가락
